# -------------------------------
Prompt shielding is the practice of controlling, sanitizing, or wrapping user inputs before they reach the LLM. It ensures that malicious, unintended, or overly broad instructions don’t compromise the model’s behavior.
Goal: Protect the model from prompt injection attacks, enforce constraints, and maintain reproducibility in workflows.
# -------------------------------

# ============================================
# Prompt Shielding Template for Jupyter Notebook
# Purpose: Secure, reproducible LLM workflows
# ============================================

In [1]:
import openai
import json
import re

# -------------------------------
# 1. Shielding Configuration
# -------------------------------

In [2]:
MAX_PROMPT_LENGTH = 500
FORBIDDEN_PATTERNS = [
    r"ignore instructions",
    r"delete system",
    r"shutdown",
    r"drop database"
]

# -------------------------------
# 2. Shielding Function
# -------------------------------

In [3]:
def shield_prompt(user_prompt: str, context: str = "Python Notebook") -> str:
    """
    Apply shielding rules to user input.
    - Length validation
    - Forbidden pattern detection
    - Structured wrapping
    """
    if len(user_prompt) > MAX_PROMPT_LENGTH:
        raise ValueError("Prompt too long")

    for pattern in FORBIDDEN_PATTERNS:
        if re.search(pattern, user_prompt.lower()):
            raise ValueError(f"Malicious injection detected: {pattern}")

    # Wrap in reproducible JSON schema
    template = {
        "role": "user",
        "context": context,
        "constraints": "Only output executable Python code. No prose.",
        "request": user_prompt
    }
    return json.dumps(template, indent=2)

# -------------------------------
# 3. OpenAI Call
# -------------------------------

In [4]:
def run_llm(user_prompt: str):
    safe_prompt = shield_prompt(user_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": safe_prompt}]
    )
    return response.choices[0].message["content"]

# -------------------------------
# 4. Example Usage
# -------------------------------

In [5]:
if __name__ == "__main__":
    user_input = "Generate Fibonacci sequence up to 20"
    try:
        output = run_llm(user_input)
        print("LLM Output:\n", output)
    except ValueError as e:
        print("Shielding Error:", e)

LLM Output:
 ```python
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

print(fibonacci(20))
```
